# 🧠 MURE x PRD-LLM: The Teacher-less Integration
**Body: PRD-LLM (Qwen 7B) | Soul: MURE SVO-CC (1M+ Rules)**

This system uses MURE as the primary reasoning engine and PRD-LLM for natural language generation. No Teacher APIs (Gemini/Groq/OpenAI) are used.

In [ ]:
# @title 1. Setup & Dependencies
!pip install transformers accelerate bitsandbytes peft fastapi uvicorn pyngrok nest-asyncio pypdf python-docx beautifulsoup4 trafilatura -q
import os, sys, torch
from google.colab import drive
drive.mount('/content/drive')
BRAIN_PATH = '/content/drive/MyDrive/svo cc brain'
os.makedirs(BRAIN_PATH, exist_ok=True)

## 🧬 Part 1: MURE Soul (Python Implementation)

In [ ]:
# @title 2. MURE Reasoner Logic
import unicodedata, re, json, time

class MyanmarProcessor:
    @staticmethod
    def is_myanmar(text): return any(0x1000 <= ord(c) <= 0x109F for c in text)
    @staticmethod
    def normalize(text): return unicodedata.normalize('NFC', text)
    @staticmethod
    def segment(text):
        r = r'[\u1000-\u1021\u1023-\u1027\u1029\u102a\u103f\u104c\u104d][\u102b-\u103e]*[\u1039\u103a]?'
        m = re.findall(r, text)
        return m if m else text.split()

class MURE:
    def __init__(self, path):
        self.path = path
        self.rules = []
        self.index = {}
        if os.path.exists(path):
            with open(path, 'r') as f: self.rules = json.load(f)
        self._reindex()

    def _reindex(self):
        self.index = {}
        for r in self.rules:
            c = r.get('cause', '').lower()
            if c not in self.index: self.index[c] = []
            self.index[c].append(r)

    def reason(self, text):
        text = MyanmarProcessor.normalize(text).lower()
        res = {'cause': text, 'effect': None, 'strength': 0}
        matches = self.index.get(text, [])
        if matches:
            best = max(matches, key=lambda x: x.get('strength', 0))
            res['effect'] = best['effect']
            res['strength'] = best['strength']
        return res

## 🤖 Part 2: PRD-LLM Body (Qwen 7B)

In [ ]:
# @title 3. Model Loader (4-bit Quantized)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')

In [ ]:
# @title 4. Merged Chat Engine
mure_soul = MURE(f'{BRAIN_PATH}/rules/rules.json')

def merged_chat(message):
    logic = mure_soul.reason(message)
    context = ""
    if logic['effect']:
        context = f"MURE Reasoning: {logic['cause']} -> {logic['effect']} (Confidence: {int(logic['strength']*100)}%)"
    
    prompt = f"<|system|>You are MURE, an AI with causal logic soul. Context: {context}<|user|>{message}<|assistant|>"
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id, **inputs, max_new_tokens=512)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(merged_chat('မိုးရွာရင် ဘာဖြစ်မလဲ'))

In [ ]:
# @title 5. API Server (Ngrok + FastAPI)
# React Web UI နဲ့ ချိတ်ဆက်ဖို့အတွက် Ngrok Auth Token ကို ဒီနေရာမှာ ထည့်ပါ။
NGROK_TOKEN = 'YOUR_NGROK_AUTH_TOKEN_HERE'  # <--- ဒီနေရာမှာ Token ပြောင်းထည့်ပါ

!pip install pyngrok nest-asyncio -q
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

if not NGROK_TOKEN or NGROK_TOKEN == 'YOUR_NGROK_AUTH_TOKEN_HERE':
    print('❌ NGROK_TOKEN ကို ထည့်ပေးပါ။')
else:
    app = FastAPI(title='MURE API Backend (PRD Merged)')
    app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

    class ChatRequest(BaseModel):
        message: str

    @app.get('/health')
    def health_check():
        return {'status': 'online', 'version': 'PRD Merged 7B LLM'}

    @app.get('/stats')
    def get_stats():
        return {'causalRules': len(mure_soul.rules)}

    @app.post('/chat')
    def chat(req: ChatRequest):
        try:
            reply = merged_chat(req.message)
            return {
                'reply': reply,
                'frame': {'effect': 'Generated via PRD Merged LLM'},
                'learned': False,
                'source': 'prd_7b_merged_llm',
                'stats': {'causalRules': len(mure_soul.rules)}
            }
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print('==========================================================')
    print(f'⭐ PUBLIC API URL: {public_url.public_url} ⭐')
    print('ဒီ URL ကို copy ကူးပြီး React Web UI ရဲ့ Settings ထဲက Python Backend API URL နေရာမှာ ထည့်ပါ။')
    print('==========================================================')

    nest_asyncio.apply()
    uvicorn.run(app, host='0.0.0.0', port=8000)
